In [ ]:
import cudf as cf
import time
import cupy as cp

import optuna
# Use native XGBoost Dask API instead of dask-ml
import xgboost as xgb
# from dask_ml.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [ ]:
# Load the processed data
print("Loading parquet data...")
df = cf.read_parquet("./tmp/unfv_with_weather_route_29_39.parquet")
print(f"Columns: {df.columns.tolist()}")

Loading parquet data...
Columns: ['route', 'trip_id', 'stopid', 'hour', 'day_of_week', 'month', 'day_of_year', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'delay_seconds', 'Climate ID', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)']


In [ ]:
df.dtypes

route                  float64
trip_id                 object
stopid                  object
hour                     int32
day_of_week              int32
month                    int32
day_of_year              int32
hour_sin               float64
hour_cos               float64
day_of_week_sin        float64
day_of_week_cos        float64
delay_seconds          float64
Climate ID             float64
Temp (°C)              float64
Dew Point Temp (°C)    float64
Rel Hum (%)            float64
Wind Spd (km/h)        float64
Visibility (km)        float64
Stn Press (kPa)        float64
dtype: object

In [ ]:
df.isna().sum()

In [4]:
df['route'] = df['route'].astype('int16')
df['trip_id'] = df['trip_id'].astype('category')
df['stopid'] = df['stopid'].astype('category')

In [5]:
df.dtypes

route                     int16
trip_id                category
stopid                 category
hour                      int32
day_of_week               int32
month                     int32
day_of_year               int32
hour_sin                float64
hour_cos                float64
day_of_week_sin         float64
day_of_week_cos         float64
delay_seconds           float64
Climate ID              float64
Temp (°C)               float64
Dew Point Temp (°C)     float64
Rel Hum (%)             float64
Wind Spd (km/h)         float64
Visibility (km)         float64
Stn Press (kPa)         float64
dtype: object

In [6]:
df.shape[0]

11287394

In [9]:
df['route'].value_counts()

route
39    6221472
29    5065922
Name: count, dtype: int64

In [10]:
df['delay_seconds'].describe()

count    1.121211e+07
mean    -1.882245e+01
std      7.378431e+02
min     -6.812163e+04
25%     -2.290000e+02
50%     -6.476200e+01
75%      8.000000e+01
max      3.779996e+04
Name: delay_seconds, dtype: float64

In [11]:
df['delay_seconds'].isna().sum()

np.int64(75282)

In [12]:
# drop na delay seconds
df = df.dropna(subset=['delay_seconds'])

In [13]:
# X = df.drop(columns=['delay_seconds', 'route'])
# y = df['delay_seconds']

In [14]:
# X.head()

In [15]:
# y.head()

In [16]:
print(f"\nFeature shape: {df.shape[0]:,} rows x {df.shape[1]} columns")


Feature shape: 11,212,112 rows x 19 columns


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['delay_seconds', 'stopid', 'trip_id']),
    df['delay_seconds'],
    test_size=0.2,
    random_state=42
)

print(f"\nTrain size: {len(X_train):,}")
print(f"Test size: {len(X_test):,}")


Train size: 8,969,689
Test size: 2,242,423


In [18]:
X_train = X_train.to_cupy()
y_train = y_train.to_cupy()
X_test = X_test.to_cupy()
y_test = y_test.to_cupy()

In [19]:
cp.get_default_memory_pool().free_all_blocks()

In [20]:
# Define Optuna objective function for hyperparameter tuning
import numpy as np
def objective(trial):
    """Optuna objective function to minimize RMSE using native XGBoost Dask API"""
    
    # Define hyperparameter search space
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 10_000,15_000, step=1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    
    n_samples = X_train.shape[0]
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    cp.get_default_memory_pool().free_all_blocks() 
    
    avg_rmse_scores = np.zeros(kf.get_n_splits())
    
    # KFold.split only needs sample count — pass a range, no GPU→CPU data transfer
    for i, (train_index, val_index) in enumerate(kf.split(range(n_samples))):
        # Index cupy arrays directly with numpy indices
        dtrain = xgb.QuantileDMatrix(X_train[train_index], label=y_train[train_index])
        dval = xgb.QuantileDMatrix(X_train[val_index], label=y_train[val_index], ref=dtrain)

        model = xgb.train(params, dtrain, evals=[(dval, 'eval')], 
                      verbose_eval=False, early_stopping_rounds=100)
    
        # Make predictions (returns numpy array)
        y_pred = model.predict(dval)
    
        # Convert only the fold's labels to numpy for RMSE calculation
        y_fold = y_train[val_index].get()
        mse = mean_squared_error(y_fold, y_pred)
        rmse = np.sqrt(mse)
        avg_rmse_scores[i] = rmse

    # Return average RMSE across folds
    return float(np.mean(avg_rmse_scores))

print("Objective function defined with native XGBoost and Optuna GPU")

Objective function defined with native XGBoost and Optuna GPU


In [21]:
# Run Optuna hyperparameter optimization
print("Starting Optuna hyperparameter optimization...")
print("XGBoost will use GPU acceleration, and Optuna will run trials sequentially to maximize GPU utilization.")
print("This may take a while...\n")


# Create study with DaskStorage
study = optuna.create_study(
    direction='minimize',  # Minimize RMSE
    study_name='xgboost_delay_prediction',
    sampler=optuna.samplers.TPESampler(seed=42),
)

# Run optimization sequentially (XGBoost Dask handles parallelization internally)
n_trials = 20  # Total number of trials

print(f"Running {n_trials} trials...")

# Run sequential optimization - each trial uses Dask cluster efficiently
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print("\n" + "="*60)
print("OPTIMIZATION COMPLETE!")
print("="*60)
print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best RMSE: {study.best_value:.4f}")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-02-21 19:32:08,068] A new study created in memory with name: xgboost_delay_prediction


Starting Optuna hyperparameter optimization...
XGBoost will use GPU acceleration, and Optuna will run trials sequentially to maximize GPU utilization.
This may take a while...

Running 20 trials...


  0%|          | 0/20 [00:00<?, ?it/s]

/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:32:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:32:09] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:32:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:32:12] WARNING: /__w/xg

[I 2026-02-21 19:32:13,619] Trial 0 finished with value: 415.2471623082007 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'n_estimators': 14000, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746}. Best is trial 0 with value: 415.2471623082007.


...

[I 2026-02-21 19:33:06,497] Trial 17 finished with value: 333.22245536233766 and parameters: {'max_depth': 10, 'learning_rate': 0.28668645969289, 'n_estimators': 15000, 'subsample': 0.7684757213625769, 'colsample_bytree': 0.6516819941479122}. Best is trial 17 with value: 333.22245536233766.


/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:33:06] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:33:07] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:33:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:33:09] WARNING: /__w/xg

[I 2026-02-21 19:33:10,464] Trial 18 finished with value: 444.37340863322004 and parameters: {'max_depth': 10, 'learning_rate': 0.08122101426437732, 'n_estimators': 15000, 'subsample': 0.7539871612489546, 'colsample_bytree': 0.8238644262750464}. Best is trial 17 with value: 333.22245536233766.


/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:33:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:33:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:33:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:33:13] WARNING: /__w/xg

[I 2026-02-21 19:33:14,279] Trial 19 finished with value: 353.22479891343187 and parameters: {'max_depth': 10, 'learning_rate': 0.1710650243602999, 'n_estimators': 12000, 'subsample': 0.651129729713056, 'colsample_bytree': 0.9258360514720573}. Best is trial 17 with value: 333.22245536233766.

OPTIMIZATION COMPLETE!

Best trial: #17
Best RMSE: 333.2225

Best hyperparameters:
  max_depth: 10
  learning_rate: 0.28668645969289
  n_estimators: 15000
  subsample: 0.7684757213625769
  colsample_bytree: 0.6516819941479122


In [30]:
study.best_params

{'max_depth': 10,
 'learning_rate': 0.28668645969289,
 'n_estimators': 15000,
 'subsample': 0.7684757213625769,
 'colsample_bytree': 0.6516819941479122}

In [31]:
import cupy as cp
cp.get_default_memory_pool().free_all_blocks()

In [32]:
# Train final model with best parameters using native XGBoost Dask API
print("Training final model with best hyperparameters...")
print("="*60)

# Get best parameters
best_params = study.best_params.copy()
best_params['objective'] = 'reg:squarederror'
best_params['tree_method'] = 'hist'  # Required for Dask

# Create QuantileDMatrix for training
dtrain = xgb.QuantileDMatrix(X_train, y_train)

# Train final model using native Dask XGBoost
output = xgb.train(
    best_params,
    dtrain,
)

Training final model with best hyperparameters...


/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [19:34:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [33]:
dtest = xgb.QuantileDMatrix(X_test)
y_pred = output.predict(dtest)

In [34]:
rmse = np.sqrt(mean_squared_error(y_test.get(), y_pred))
r2 = r2_score(y_test.get(), y_pred)
print("Final Model Performance on Test Set:")
print(f"  RMSE: {rmse:.4f}")
print(f"  R²: {r2:.4f}")

Final Model Performance on Test Set:
  RMSE: 403.5915
  R²: 0.7115
